按 Market 删除保留期外且已不存在于 t_master_consumer 的 CBR 相关数据。

删除条件（每个 Market 独立计算）：
1. 增量三张 CBR t表（按 MarketCode 过滤）UNION 该 Market 备份的 cbrl/cbrlpublic/cbrldrjart（仅 KOR 有 drjart）
2. 按 MarketCode+MDMKey 分组取 UpdatedTimestamp 最新一条
3. 筛选 UpdatedTimestamp < 当前日期 - retention_days（NULL 保留不删）
4. MarketCode+MDMKey 在 t_master_consumer（scon_mrkt_code+consumermdmkey）中不存在

删除目标（按 MarketCode+MDMKey 匹配）：
- derived 表 l1/l2/l3（scon_mrkt_code+consumermdmkey）
- 增量 CBR c表 & t表 共 6 张
- 备份 cbrl/cbrlpublic/cbrldrjart（abfss 路径）

In [0]:
from functools import reduce
from delta.tables import DeltaTable
from pyspark.sql import DataFrame, functions as F

In [0]:
dbutils.widgets.text("master_db", "catalog_southeastasia_mdm_golden_prod.consumer_master")
dbutils.widgets.text("combine_db", "catalog_southeastasia_mdm_golden_prod.consumer_combine")
dbutils.widgets.text("backup_prefix", "abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_prod/history_data_loading/Prod_Consumer_20260715")
dbutils.widgets.text("retention_days", str(365 * 6))
dbutils.widgets.text("markets", "AUS,HKG,IDN,JPN,KOR,MYS,NZL,PHL,SGP,THA,TWN,VNM")

master_db = dbutils.widgets.get("master_db")
combine_db = dbutils.widgets.get("combine_db")
backup_prefix = dbutils.widgets.get("backup_prefix").rstrip("/")
retention_days = int(dbutils.widgets.get("retention_days"))
markets = [m.strip().upper() for m in dbutils.widgets.get("markets").split(",") if m.strip()]

print(f"master_db: {master_db}")
print(f"combine_db: {combine_db}")
print(f"backup_prefix: {backup_prefix}")
print(f"retention_days: {retention_days}")
print(f"markets({len(markets)}): {markets}")

In [0]:
CBR_DATASETS = ["cbr_dataset", "cbr_withoutpii_dataset", "cbrdrjart_dataset"]
BACKUP_TABLES = ["cbrl", "cbrlpublic", "cbrldrjart"]
DERIVED_TABLES = ["t_derived_consumer_l1", "t_derived_consumer_l2", "t_derived_consumer_l3"]
KEY_COLS = ["MarketCode", "MDMKey"]


def backup_path(market, table):
    return f"{backup_prefix}/{market.upper()}/{market.lower()}_elcconsumermdm/{table}"

In [0]:
def find_delete_keys(market):
    """
    计算某 Market 要删除的 (MarketCode, MDMKey) 集合：
    增量 t表 UNION 备份表 -> 按 key 取最新 UpdatedTimestamp -> 过期 -> master 中不存在。
    """
    cols = KEY_COLS + ["UpdatedTimestamp"]

    dfs = [
        spark.table(f"{combine_db}.t_{t}").where(F.col("MarketCode") == market).select(*cols)
        for t in CBR_DATASETS
    ]
    for t in BACKUP_TABLES:
        if t == "cbrldrjart" and market != "KOR":
            continue
        dfs.append(spark.read.format("delta").load(backup_path(market, t)).select(*cols))

    expired_keys = (
        reduce(DataFrame.unionAll, dfs)
        .groupBy(*KEY_COLS)
        .agg(F.max("UpdatedTimestamp").alias("UpdatedTimestamp"))
        .where(F.col("UpdatedTimestamp") < F.date_sub(F.current_date(), retention_days))
    )

    master_keys = (
        spark.table(f"{master_db}.t_master_consumer")
        .where(F.col("scon_mrkt_code") == market)
        .select(
            F.col("scon_mrkt_code").alias("MarketCode"),
            F.col("consumermdmkey").alias("MDMKey"),
        )
        .distinct()
    )

    return expired_keys.join(master_keys, KEY_COLS, "left_anti").select(*KEY_COLS)

In [0]:
def delete_by_name(table_name, keys_df, condition):
    try:
        (
            DeltaTable.forName(spark, table_name).alias("target")
            .merge(keys_df.alias("source"), condition)
            .whenMatchedDelete()
            .execute()
        )
        print(f"  processed {table_name}")
    except Exception as e:
        raise RuntimeError(f"delete failed on {table_name}: {e}") from e


def delete_by_path(table_path, keys_df, condition):
    try:
        (
            DeltaTable.forPath(spark, table_path).alias("target")
            .merge(keys_df.alias("source"), condition)
            .whenMatchedDelete()
            .execute()
        )
        print(f"  processed {table_path}")
    except Exception as e:
        raise RuntimeError(f"delete failed on {table_path}: {e}") from e

In [0]:
def process_market(market):
    print(f"[{market}] calculating delete keys")
    keys_df = find_delete_keys(market).persist()
    keys_count = keys_df.count()
    print(f"[{market}] {keys_count} keys to delete")

    if keys_count == 0:
        keys_df.unpersist()
        print(f"[{market}] nothing to delete, skip")
        return

    mdm_cond = "target.MarketCode = source.MarketCode AND target.MDMKey = source.MDMKey"
    derived_cond = "target.scon_mrkt_code = source.MarketCode AND target.consumermdmkey = source.MDMKey"

    print(f"[{market}] deleting derived tables")
    for t in DERIVED_TABLES:
        delete_by_name(f"{master_db}.{t}", keys_df, derived_cond)

    print(f"[{market}] deleting incremental cbr c/t tables")
    for t in CBR_DATASETS:
        for prefix in ("c_", "t_"):
            delete_by_name(f"{combine_db}.{prefix}{t}", keys_df, mdm_cond)

    print(f"[{market}] deleting backup cbr tables")
    for t in BACKUP_TABLES:
        if t == "cbrldrjart" and market != "KOR":
            continue
        delete_by_path(backup_path(market, t), keys_df, mdm_cond)

    keys_df.unpersist()
    print(f"[{market}] completed")

In [0]:
failed_markets = []
for market in markets:
    try:
        process_market(market)
    except Exception as e:
        failed_markets.append(market)
        print(f"[{market}] FAILED: {type(e).__name__}: {e}")

if failed_markets:
    raise RuntimeError(f"failed markets: {failed_markets}")
print("[all markets completed]")